# Day 56 — Orchestration basics: Prefect intro
Objectives:
- Create simple Prefect flows and tasks.
- Parameterize a pipeline (preprocess → train → evaluate).
- Run locally and observe logs.
Note: `pip install prefect` (already added to requirements.txt).


In [ ]:
from prefect import flow, task

@task
def load_data():
    import seaborn as sns
    df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','fare','age'])
    return df

@task
def preprocess(df):
    import pandas as pd
    df = df.copy()
    df['fare'] = pd.to_numeric(df['fare'], errors='coerce').fillna(df['fare'].median())
    df = df.dropna(subset=['age'])
    return df

@task
def train(df):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from pathlib import Path

    import joblib
    X = df[['sex','class','fare','age']]; y = df['survived']
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                             ('num', StandardScaler(), ['fare','age'])])
    pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
    Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
    pipe.fit(Xtr,ytr)
    import numpy as np
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    artifact_dir = Path('artifacts/day56')
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, artifact_dir / 'prefect_titanic_pipeline.joblib')
    return auc

@flow
def training_flow():
    df = load_data()
    df2 = preprocess(df)
    auc = train(df2)
    print('AUC:', auc)

# Run the flow
if __name__ == '__main__':
    training_flow()


## Learner exercises and progressive hints

1. Add `test_size` and `random_state` parameters to the training flow.
2. Split the training task into separate train and evaluate tasks with explicit
   outputs.
3. Explore the optional local Prefect UI and scheduling basics.

### Progressive hints

1. Pass parameters from the flow to the task; log them with the resulting metric
   so a run can be reproduced.
2. Return the fitted pipeline plus held-out arrays or a small typed result.
   Consider whether passing a large dataset between task processes would scale.
3. First prove `training_flow()` succeeds directly. Then run a local server and
   use Prefect 3's current `serve`/deployment workflow—not commands copied from
   older major versions.

The separate solution reinforces retry, notification, and scheduling concepts.
Use the Prefect 3 execution path in this guide and the current learner
environment when translating those concepts.

### Additional mastery practice

Orchestrate explicit, typed tasks whose retries are safe, artifacts are versioned, and failures are observable. A flow wrapper does not repair an unsafe task.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Retry and idempotence:** Add retries to a task that writes an artifact. Make the write idempotent so a failure after writing cannot create duplicate or partially valid outputs.
   **Progressive hint:** Write to a temporary path, validate, then atomically replace a versioned destination. A retry should produce the same logical result.
5. **Cache-key design:** Design a task cache key that changes when data fingerprint, code/config, or relevant parameters change, but not when an unrelated log message changes.
   **Progressive hint:** Hash canonical semantic inputs and include a task/schema version. Do not cache a task whose hidden external state is untracked.
6. **Failure observability:** Instrument a three-task flow so logs and a final summary identify run ID, task, safe input version, attempt, elapsed time, artifact ID, and failure category without logging sensitive rows.
   **Progressive hint:** Use structured fields and task/run context. Emit counts and opaque IDs rather than raw feature values.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Retry and idempotence


# Practice 5 — Cache-key design


# Practice 6 — Failure observability
